# Notebook 02 — Entrenamiento Contrastivo (SupCon)

**Objetivo:** Entrenar el encoder ResNet18 con Supervised Contrastive Learning para producir embeddings de 1024 dims.

**IMPORTANTE:** Este notebook está diseñado para ejecutarse en **Google Colab con GPU**.
En CPU local toma ~2h por epoch con batch_size=32. En Colab T4 toma ~2-5 min por epoch con batch_size=256.

## Cómo ejecutar en Colab
1. Runtime → Change runtime type → **T4 GPU** (gratis) o A100 (Pro)
2. Descomenta las celdas marcadas `# COLAB`
3. Runtime → Run all
4. Al terminar: descarga `artifacts/checkpoints/encoder_best.pt` para continuar localmente

In [1]:
# ════════════════════════════════════════════════════════════════
# SETUP — Colab sin auto-push (clone público con destino absoluto)
# ════════════════════════════════════════════════════════════════
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_DIR = f"/content/{REPO}"
    REPO_URL = "https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning.git"

    os.chdir("/content")

    for stray in (f"/{REPO}", REPO_DIR):
        if os.path.exists(stray) and not os.path.exists(f"{stray}/.git"):
            shutil.rmtree(stray, ignore_errors=True)

    if not os.path.exists(f"{REPO_DIR}/.git"):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")

    assert os.path.exists(f"{REPO_DIR}/.git"), f"git clone falló: {REPO_DIR}"

    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo local.")

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

/content/Malaria-Dectetion-Deeplearning
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 368 bytes | 368.00 KiB/s, done.
From https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning
 * branch            main       -> FETCH_HEAD
   066f115..f4d6869  main       -> origin/main
Updating 066f115..f4d6869
Fast-forward
 configs/contrastive.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Colab listo. Pesados → Drive, ligeros → repo local.
Working dir: /content/Malaria-Dectetion-Deeplearning


In [2]:
# ════════════════════════════════════════════════════════════════
# DATASET — Kaggle API (solo si falta cell_images/)
# ════════════════════════════════════════════════════════════════
if IN_COLAB and not os.path.exists("cell_images"):
    if os.path.exists(f"{DRIVE_ROOT}/kaggle.json"):
        get_ipython().system("mkdir -p ~/.kaggle")
        get_ipython().system(f"cp {DRIVE_ROOT}/kaggle.json ~/.kaggle/")
    else:
        from google.colab import files
        print("Sube tu kaggle.json (Kaggle → Settings → Create API Token):")
        files.upload()
        get_ipython().system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/")
        get_ipython().system(f"cp ~/.kaggle/kaggle.json {DRIVE_ROOT}/kaggle.json")
    get_ipython().system("chmod 600 ~/.kaggle/kaggle.json")
    get_ipython().system("pip install kaggle -q")
    get_ipython().system("kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -q")
    get_ipython().system("unzip -q cell-images-for-detecting-malaria.zip")
    get_ipython().system("rm -f cell-images-for-detecting-malaria.zip")
    get_ipython().system("if [ -d cell_images/cell_images ]; then mv cell_images/cell_images/* cell_images/ 2>/dev/null; rmdir cell_images/cell_images 2>/dev/null; fi")
    print(f"✓ Dataset listo: {len(os.listdir('cell_images/Parasitized'))} parasitized, "
          f"{len(os.listdir('cell_images/Uninfected'))} uninfected")

In [3]:
import sys
from pathlib import Path
_p = Path().resolve()
sys.path.insert(0, str(_p if (_p / 'src').exists() else _p.parent))

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_json
from src.data.split import make_stratified_split
from src.data.augmentations import get_contrastive_transform
from src.data.dataset import SupConPairDataset
from src.models.encoder import ContrastiveEncoder
from src.training.train_contrastive import train
from src.visualization.training_plots import plot_training_curves

import matplotlib.pyplot as plt
%matplotlib inline

set_global_seed(42)

In [5]:
cfg = load_config('configs/contrastive.yaml')
data_cfg = load_config('configs/data.yaml')
print('Configuración del encoder:', cfg['encoder'])
print('Épocas:', cfg['training']['epochs'])

Configuración del encoder: {'embedding_dim': 1024, 'proj_dim': 128, 'pretrained': True, 'backbone': 'resnet18'}
Épocas: 50


## 1. Preparar datos

In [6]:
from torch.utils.data import DataLoader

# Generar splits si no existen
train_df, val_df, test_df = make_stratified_split(
    dataset_root=data_cfg['dataset_root'],
    processed_dir=data_cfg['processed_dir'],
    seed=data_cfg['seed'],
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

use_gpu = torch.cuda.is_available()
aug_cfg = cfg.get('augmentations', {})
transform = get_contrastive_transform(
    img_size=aug_cfg.get('img_size', 96),
    blur_prob=aug_cfg.get('blur_prob', 0.3),
)

batch_size = cfg['training']['batch_size_gpu'] if use_gpu else cfg['training']['batch_size_cpu']
num_workers = 2 if use_gpu else 0
print(f'batch_size={batch_size} | num_workers={num_workers} | GPU={use_gpu}')

train_ds = SupConPairDataset('data/processed/train.csv', transform=transform)
val_ds   = SupConPairDataset('data/processed/val.csv',   transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=use_gpu, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=use_gpu)

Train: 19290 | Val: 4133 | Test: 4135
batch_size=256 | num_workers=2 | GPU=True


## 2. Entrenamiento SupCon

In [7]:
from pathlib import Path
Path('artifacts/checkpoints').mkdir(parents=True, exist_ok=True)

history = train(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir='artifacts/checkpoints',
)
print('\nEntrenamiento completado!')
print(f'Mejor val_loss: {min(history["val"]):.4f} (epoch {history["val"].index(min(history["val"]))+1})')

2026-05-26 03:55:05 [INFO] train_contrastive: GPU detectada: Tesla T4


INFO:train_contrastive:GPU detectada: Tesla T4
/content/Malaria-Dectetion-Deeplearning/src/training/train_contrastive.py:126: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
/content/Malaria-Dectetion-Deeplearning/src/training/train_contrastive.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


2026-05-26 03:56:53 [INFO] train_contrastive: Epoch   1/50 | train_loss=5.8751 | val_loss=5.7041 | lr=0.000999


INFO:train_contrastive:Epoch   1/50 | train_loss=5.8751 | val_loss=5.7041 | lr=0.000999


2026-05-26 03:56:53 [INFO] train_contrastive:   ✓ Nuevo mejor modelo guardado (val_loss=5.7041)


...

2026-05-26 05:02:42 [INFO] train_contrastive: Early stopping en epoch 37


INFO:train_contrastive:Early stopping en epoch 37


2026-05-26 05:02:42 [INFO] train_contrastive: Entrenamiento completado. Mejor val_loss: 5.5883


INFO:train_contrastive:Entrenamiento completado. Mejor val_loss: 5.5883



Entrenamiento completado!
Mejor val_loss: 5.5883 (epoch 30)


In [8]:
fig = plot_training_curves(history, save_path='artifacts/figures/training_curves.png')
plt.show()

import json
Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
with open('artifacts/logs/contrastive_history.json', 'w') as f:
    json.dump(history, f)

In [9]:
# En Colab, encoder_best.pt YA está en Drive vía el simlink
# (artifacts/checkpoints/ → /content/drive/MyDrive/malaria_project/checkpoints/)
# El NB03 lo leerá automáticamente desde la misma ruta.
if IN_COLAB:
    ckpt = Path("artifacts/checkpoints/encoder_best.pt")
    if ckpt.exists():
        size_mb = ckpt.stat().st_size / 1e6
        print(f"✓ Checkpoint en Drive: {ckpt.resolve()} ({size_mb:.1f} MB)")
    else:
        print("⚠ No se encontró encoder_best.pt — ¿terminó el entrenamiento?")

✓ Checkpoint en Drive: /content/drive/MyDrive/malaria_project/checkpoints/encoder_best.pt (147.7 MB)


In [10]:
# COLAB — descomenta para descargar el checkpoint a tu PC
# from google.colab import files
# files.download('artifacts/checkpoints/encoder_best.pt')

# O guardar en Drive:
import shutil
shutil.copy('artifacts/checkpoints/encoder_best.pt',
            '/content/drive/MyDrive/malaria_encoder_best.pt')

'/content/drive/MyDrive/malaria_encoder_best.pt'

In [ ]:
# ════════════════════════════════════════════════════════════════
# DESCARGA MANUAL — baja el .ipynb ejecutado a tu PC
# Sin auto-push: tú subes/entregas el notebook por el medio que prefieras.
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "02_contrastive_training"
    # Forzar guardado del .ipynb (preserva outputs y figuras embebidas)
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    # Descargar a tu PC (revisa carpeta de descargas)
    from google.colab import files
    files.download(f"notebooks/{NOTEBOOK}.ipynb")
    print(f"✓ Descarga iniciada: {NOTEBOOK}.ipynb")

## Siguiente paso
Coloca `encoder_best.pt` en `artifacts/checkpoints/` y ejecuta:
```bash
python -m scripts.extract_embeddings --checkpoint artifacts/checkpoints/encoder_best.pt
```
O abre `notebooks/03_extract_embeddings.ipynb`.